In [1]:
import sqlite3
import pandas as pd
import random
from datetime import datetime, timedelta

print("✅ Libraries loaded")

✅ Libraries loaded


In [2]:
random.seed(42)

categories = ['Furniture', 'Technology', 'Office Supplies']
sub_cats = {
    'Furniture': ['Chairs', 'Tables', 'Bookcases', 'Furnishings'],
    'Technology': ['Phones', 'Computers', 'Accessories', 'Copiers'],
    'Office Supplies': ['Paper', 'Binders', 'Storage', 'Art', 'Fasteners']
}
regions = ['West', 'East', 'Central', 'South']
segments = ['Consumer', 'Corporate', 'Home Office']
ship_modes = ['Standard Class', 'Second Class', 'First Class', 'Same Day']

products = [
    ('TEC-001','Samsung Galaxy Tab','Technology','Phones'),
    ('TEC-002','Apple MacBook Pro','Technology','Computers'),
    ('TEC-003','Logitech Wireless Mouse','Technology','Accessories'),
    ('TEC-004','Canon Printer','Technology','Copiers'),
    ('FUR-001','IKEA Office Chair','Furniture','Chairs'),
    ('FUR-002','Oak Dining Table','Furniture','Tables'),
    ('FUR-003','Wooden Bookcase','Furniture','Bookcases'),
    ('FUR-004','Desk Lamp','Furniture','Furnishings'),
    ('OFF-001','A4 Paper Pack','Office Supplies','Paper'),
    ('OFF-002','3-Ring Binder','Office Supplies','Binders'),
    ('OFF-003','Storage Box','Office Supplies','Storage'),
    ('OFF-004','Sketch Pads','Office Supplies','Art'),
]

prices = {
    'TEC-001':399,'TEC-002':1299,'TEC-003':49,'TEC-004':499,
    'FUR-001':299,'FUR-002':699,'FUR-003':199,'FUR-004':89,
    'OFF-001':29,'OFF-002':19,'OFF-003':39,'OFF-004':15,
}

states = ['California','Texas','New York','Florida','Illinois',
          'Pennsylvania','Ohio','Georgia','Michigan','North Carolina']

rows = []
base_date = datetime(2022, 1, 1)

for i in range(1, 3001):
    prod = random.choice(products)
    region = random.choice(regions)
    state = random.choice(states)
    segment = random.choice(segments)
    ship_mode = random.choice(ship_modes)
    order_date = base_date + timedelta(days=random.randint(0, 730))
    qty = random.randint(1, 10)
    price = prices[prod[0]]
    discount = round(random.choice([0, 0, 0, 0.1, 0.2, 0.3]), 2)
    sales = round(price * qty * (1 - discount), 2)
    profit = round(sales * random.uniform(0.05, 0.35), 2)

    rows.append((
        f'ORD-{i:04d}', f'CUST-{random.randint(1,500):04d}',
        order_date.strftime('%Y-%m-%d'),
        (order_date + timedelta(days=random.randint(1,7))).strftime('%Y-%m-%d'),
        ship_mode, segment, state, region,
        prod[0], prod[1], prod[2], prod[3],
        qty, discount, sales, profit
    ))

print("✅ Dataset generated:", len(rows), "rows")

✅ Dataset generated: 3000 rows


In [3]:
conn = sqlite3.connect('superstore.db')
cur = conn.cursor()

cur.execute('DROP TABLE IF EXISTS orders')

cur.execute('''
CREATE TABLE orders (
    order_id TEXT,
    customer_id TEXT,
    order_date TEXT,
    ship_date TEXT,
    ship_mode TEXT,
    segment TEXT,
    state TEXT,
    region TEXT,
    product_id TEXT,
    product_name TEXT,
    category TEXT,
    sub_category TEXT,
    quantity INTEGER,
    discount REAL,
    sales REAL,
    profit REAL
)
''')

cur.executemany('INSERT INTO orders VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)', rows)
conn.commit()

print("✅ Database created with 3000 rows")

✅ Database created with 3000 rows


In [4]:
def run_query(title, query):
    print("\n" + "="*60)
    print(title)
    print("="*60)
    df = pd.read_sql_query(query, conn)
    display(df)

In [5]:
run_query("Q1 — Revenue & Profit by Category", '''
SELECT
    category,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(AVG(profit/sales)*100,1) AS profit_margin_pct
FROM orders
GROUP BY category
ORDER BY total_sales DESC
''')


Q1 — Revenue & Profit by Category


,category,total_sales,total_profit,profit_margin_pct
0,Technology,2939514.9,573072.00,19.8
1,Furniture,1486148.8,295503.25,19.5
2,Office Supplies,123999.4,24636.62,20.1


In [6]:
run_query("Q2 — Top 5 Most Profitable Products", '''
SELECT
    product_name,
    category,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    SUM(quantity) AS units_sold
FROM orders
GROUP BY product_name, category
ORDER BY total_profit DESC
LIMIT 5
''')


Q2 — Top 5 Most Profitable Products


,product_name,category,total_sales,total_profit,units_sold
0,Apple MacBook Pro,Technology,1706756.1,325385.54,1459
1,Oak Dining Table,Furniture,770577.6,157881.19,1240
2,Canon Printer,Technology,697552.1,143415.28,1553
3,Samsung Galaxy Tab,Technology,475887.3,92457.01,1306
4,IKEA Office Chair,Furniture,338617.5,64300.09,1259


In [7]:
run_query("Q3 — Monthly Revenue Trend", '''
SELECT
    STRFTIME('%Y', order_date) AS year,
    STRFTIME('%m', order_date) AS month,
    ROUND(SUM(sales), 2) AS monthly_sales,
    COUNT(DISTINCT order_id) AS orders_count
FROM orders
GROUP BY year, month
ORDER BY year, month
''')


Q3 — Monthly Revenue Trend


,year,month,monthly_sales,orders_count
0,2022,01,207001.8,127
1,2022,02,136787.7,107
2,2022,03,166833.8,138
3,2022,04,180090.6,131
4,2022,05,209268.6,147
5,2022,06,232479.3,122
6,2022,07,173643.3,128
7,2022,08,197373.6,132
8,2022,09,181478.6,147
9,2022,10,146797.1,119


In [8]:
run_query("Q4 — Region Performance", '''
SELECT
    region,
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(sales)/COUNT(DISTINCT order_id), 2) AS avg_order_value
FROM orders
GROUP BY region
ORDER BY total_sales DESC
''')


Q4 — Region Performance


,region,unique_customers,total_orders,total_sales,total_profit,avg_order_value
0,South,383,763,1199205.9,230436.61,1571.70
1,West,392,753,1182821.6,225473.86,1570.81
2,East,393,734,1112879.7,226676.91,1516.18
3,Central,382,750,1054755.9,210624.49,1406.34


In [9]:
run_query("Q5 — Customer Segment Analysis", '''
SELECT
    segment,
    COUNT(DISTINCT customer_id) AS customers,
    ROUND(SUM(sales), 2) AS revenue,
    ROUND(AVG(sales), 2) AS avg_sale,
    ROUND(SUM(profit), 2) AS profit
FROM orders
GROUP BY segment
ORDER BY revenue DESC
''')


Q5 — Customer Segment Analysis


,segment,customers,revenue,avg_sale,profit
0,Home Office,427,1556068.5,1554.51,304704.13
1,Corporate,431,1521613.5,1533.88,300720.00
2,Consumer,437,1471981.1,1461.75,287787.74


In [10]:
run_query("Q6 — Shipping Mode Efficiency", '''
WITH shipping_stats AS (
    SELECT
        ship_mode,
        AVG(JULIANDAY(ship_date) - JULIANDAY(order_date)) AS avg_days,
        COUNT(*) AS shipments,
        ROUND(SUM(sales), 2) AS revenue
    FROM orders
    GROUP BY ship_mode
)
SELECT
    ship_mode,
    ROUND(avg_days, 1) AS avg_ship_days,
    shipments,
    revenue
FROM shipping_stats
ORDER BY avg_days
''')


Q6 — Shipping Mode Efficiency


,ship_mode,avg_ship_days,shipments,revenue
0,First Class,4.0,740,1054322.4
1,Same Day,4.0,759,1112023.2
2,Standard Class,4.0,729,1131173.9
3,Second Class,4.1,772,1252143.6


In [11]:
run_query("Q7 — Discount Impact on Profit", '''
SELECT
    CASE
        WHEN discount = 0 THEN 'No Discount'
        WHEN discount <= 0.1 THEN '1-10%'
        WHEN discount <= 0.2 THEN '11-20%'
        ELSE '20%+'
    END AS discount_tier,
    COUNT(*) AS orders,
    ROUND(AVG(profit), 2) AS avg_profit,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(AVG(profit/sales)*100,1) AS profit_margin_pct
FROM orders
GROUP BY discount_tier
ORDER BY avg_profit DESC
''')


Q7 — Discount Impact on Profit


,discount_tier,orders,avg_profit,total_profit,profit_margin_pct
0,No Discount,1470,323.87,476087.57,20.0
1,1-10%,549,309.46,169894.60,19.6
2,11-20%,503,288.20,144965.52,19.6
3,20%+,478,213.94,102264.18,19.8


In [12]:
run_query("Q8 — State Rankings", '''
SELECT
    state,
    region,
    ROUND(SUM(sales), 2) AS total_sales,
    RANK() OVER (ORDER BY SUM(sales) DESC) AS sales_rank
FROM orders
GROUP BY state, region
ORDER BY sales_rank
LIMIT 10
''')


Q8 — State Rankings


,state,region,total_sales,sales_rank
0,Michigan,South,162729.2,1
1,Pennsylvania,South,162437.6,2
2,California,West,148090.3,3
3,Georgia,East,146698.5,4
4,Florida,Central,140442.1,5
5,Florida,South,138847.3,6
6,Ohio,West,136071.3,7
7,North Carolina,West,131949.5,8
8,Georgia,West,130443.2,9
9,Illinois,East,126977.5,10


In [13]:
run_query("Q9 — Year-over-Year Growth", '''
WITH yearly AS (
    SELECT
        category,
        STRFTIME('%Y', order_date) AS yr,
        SUM(sales) AS sales
    FROM orders
    GROUP BY category, yr
),
pivoted AS (
    SELECT
        category,
        MAX(CASE WHEN yr='2022' THEN sales END) AS sales_2022,
        MAX(CASE WHEN yr='2023' THEN sales END) AS sales_2023
    FROM yearly
    GROUP BY category
)
SELECT
    category,
    ROUND(sales_2022, 2) AS sales_2022,
    ROUND(sales_2023, 2) AS sales_2023,
    ROUND((sales_2023 - sales_2022)/sales_2022 * 100, 1) AS yoy_growth_pct
FROM pivoted
ORDER BY yoy_growth_pct DESC
''')


Q9 — Year-over-Year Growth


,category,sales_2022,sales_2023,yoy_growth_pct
0,Technology,1408371.9,1531143.0,8.7
1,Furniture,742681.8,743467.0,0.1
2,Office Supplies,62217.7,61539.6,-1.1


In [14]:
run_query("Q10 — Top Customers", '''
SELECT
    customer_id,
    COUNT(DISTINCT order_id) AS frequency,
    ROUND(SUM(sales), 2) AS lifetime_value,
    ROUND(AVG(sales), 2) AS avg_order_value,
    MAX(order_date) AS last_order_date
FROM orders
GROUP BY customer_id
ORDER BY lifetime_value DESC
LIMIT 10
''')


Q10 — Top Customers


,customer_id,frequency,lifetime_value,avg_order_value,last_order_date
0,CUST-0146,11,32698.6,2972.60,2023-11-18
1,CUST-0005,11,32482.4,2952.95,2023-11-24
2,CUST-0066,15,31918.5,2127.90,2023-10-25
3,CUST-0295,18,29227.4,1623.74,2023-10-29
4,CUST-0046,8,27851.0,3481.38,2023-12-12
5,CUST-0017,10,27250.9,2725.09,2023-12-09
6,CUST-0003,11,26952.7,2450.25,2023-11-21
7,CUST-0314,12,26902.8,2241.90,2023-07-14
8,CUST-0111,5,26455.7,5291.14,2023-11-24
9,CUST-0186,4,25794.6,6448.65,2023-08-08


In [15]:
conn.close()
print("✅ Done!")

✅ Done!
